# "THE PRICE IS RIGHT" — Day 3

## Week 6 capstone: predict a product's price from its description

### Order of play
- DAY 1: Data Curation
- DAY 2: Data Pre-processing
- **DAY 3: Evaluation, Baselines, Traditional ML** ← *we are here*
- DAY 4: Deep Learning and LLMs
- DAY 5: Fine-tuning a Frontier Model

### Today — Evaluation, Baselines, Traditional ML
Before reaching for deep learning, we set up the **scoreboard** and a few honest **baselines**.
First a couple of trivial models (random guess, always-the-average) to anchor expectations, then
real traditional-ML models — linear regression on hand-built features, then on bag-of-words text
features, then Random Forest and XGBoost. Every model is scored the same way via
`pricer.evaluator.evaluate`, so the numbers are directly comparable.

In [5]:
import random

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor

from pricer.items import Item
from pricer.evaluator import evaluate

### Load the pre-processed dataset

This is the Day-2 output: products rewritten into clean, standard-format `summary` text. Set
`username` to your own HuggingFace username, or use `"ed-donner"` for the official copy.

In [6]:
LITE_MODE = False

In [7]:
username = "marcolerma"  # official PRE-PROCESSED copy; switch to your own once you've run Day 2
# Note: load the pre-processed `items_*` dataset (Day 2 output), NOT the Day-1 `items_raw_*` one —
# Day 3 learns from the `summary` field, which only exists after Day 2.
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


## Baseline 1 — random

The dumbest possible model: guess a random price between \$1 and \$999. This isn't meant to be
good — it sets the floor. Any real model has to beat *this* comfortably. Notice the shape of the
`evaluate` call: a predictor is just a function `item -> price`.

In [6]:
def random_pricer(item):
    return random.randrange(1, 1000)

In [7]:
random.seed(42)
evaluate(random_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$436 $166 $60 $690 $4 $525 $1 $14 $215 $229 $20 $380 $894 $51 $660 $57 $618 $68 $489 $486 $299 $91 $54 $79 $108 $214 $23 $597 $71 $495 $184 $674 $506 $639 $444 $389 $166 $404 $390 $247 $629 $811 $28 $523 $606 $139 $697 $401 $204 $83 $125 $91 $506 $662 $292 $29 $88 $350 $102 $356 $632 $275 $528 $161 $198 $45 $508 $26 $425 $52 $965 $928 $108 $62 $516 $265 $726 $634 $617 $888 $763 $359 $562 $98 $695 $12 $135 $372 $114 $746 $275 $13 $856 $204 $880 $124 $356 $128 $195 $76 $755 $257 $130 $241 $115 $97 $662 $126 $688 $661 $640 $350 $43 $496 $456 $6 $522 $727 $195 $65 $458 $300 $207 $782 $456 $645 $551 $126 $523 $194 $793 $650 $710 $28 $170 $827 $172 $755 $196 $273 $211 $182 $137 $916 $772 $549 $853 $226 $58 $185 $653 $406 $359 $766 $918 $380 $421 $91 $48 $93 $36 $737 $533 $539 $570 $742 $151 $389 $880 $588 $384 $349 $195 $119 $420 $357 $66 $681 $45 $789 $541 $132 $343 $85 $762 $675 $300 $544 $6 $368 $356 $560 $445 $403 $158 $915 $397 $602 $915 $14 

## Baseline 2 — always guess the average

Slightly less silly: always predict the *average training price*. A constant predictor like this
is a surprisingly stubborn baseline — it's what any feature-based model must beat to justify its
features.

In [8]:
training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item):
    return training_average

140.56967545


In [9]:
evaluate(constant_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$78 $25 $86 $71 $111 $89 $4 $75 $105 $189 $572 $238 $121 $86 $61 $108 $61 $91 $70 $22 $7 $17 $56 $34 $191 $312 $354 $121 $42 $61 $121 $81 $19 $60 $25 $678 $81 $85 $73 $103 $59 $61 $106 $114 $79 $116 $123 $109 $5 $61 $105 $11 $334 $21 $87 $6 $134 $101 $62 $129 $95 $63 $50 $31 $488 $51 $99 $304 $16 $65 $109 $124 $139 $122 $91 $105 $16 $131 $124 $122 $21 $129 $111 $42 $114 $81 $42 $165 $21 $95 $119 $46 $121 $106 $132 $88 $107 $17 $129 $434 $41 $24 $104 $2 $108 $23 $116 $259 $110 $158 $81 $174 $110 $12 $55 $29 $116 $121 $85 $38 $125 $52 $70 $25 $59 $81 $121 $42 $38 $2 $69 $3 $55 $111 $76 $126 $64 $71 $12 $2 $76 $109 $60 $121 $53 $109 $96 $369 $124 $108 $122 $35 $94 $1 $121 $138 $92 $85 $179 $91 $148 $115 $99 $128 $699 $118 $307 $91 $101 $131 $116 $119 $279 $118 $39 $8 $113 $48 $46 $48 $514 $116 $159 $108 $91 $119 $7 $74 $81 $114 $106 $90 $106 $2 $41 $61 $29 $140 $90 $115 

## Traditional ML 1 — linear regression on simple features

Now an actual model. We hand-build a tiny feature vector for each item — its weight, a flag for
whether the weight is missing, and the length of its summary text — and fit a linear regression.
These features are crude, so don't expect much; the point is to see the machinery and the
evaluation working end to end.

In [10]:
def get_features(item):
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight == 0 else 0,
        "text_length": len(item.summary),
    }

In [14]:
def list_to_dataframe(items):
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df["price"] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

TypeError: object of type 'NoneType' has no len()

In [13]:
# Traditional Linear Regression
np.random.seed(42)

feature_columns = ["weight", "weight_unknown", "text_length"]

X_train = train_df[feature_columns]
y_train = train_df["price"]
X_test = test_df[feature_columns]
y_test = test_df["price"]

model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and report sklearn's own metrics
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

NameError: name 'train_df' is not defined

In [ ]:
def linear_regression_pricer(item):
    features = get_features(item)
    features_df = pd.DataFrame([features])
    return model.predict(features_df)[0]

In [15]:
evaluate(linear_regression_pricer, test)

NameError: name 'linear_regression_pricer' is not defined

## Traditional ML 2 — linear regression on words (bag-of-words)

Hand-built features ignore *what the product actually is*. So let's feed the model the text.
`CountVectorizer` turns each summary into a vector of word counts over the 2,000 most common words
(dropping English stop-words). It's the classic "bag of words" representation — order is thrown
away, but word presence/frequency carries a lot of price signal (think "platinum" vs "plastic").

In [16]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [17]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X = vectorizer.fit_transform(documents)

AttributeError: 'NoneType' object has no attribute 'lower'

In [ ]:
# A peek at some of the words the vectorizer selected (not including stop words)
selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

In [ ]:
regressor = LinearRegression()
regressor.fit(X, prices)

In [ ]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)  # clamp negatives to 0

In [ ]:
evaluate(natural_language_linear_regression_pricer, test)

## Traditional ML 3 — Random Forest

The **Random Forest** is an *ensemble*: it combines many simple **decision trees**. A single
decision tree predicts by walking a flow-chart of IF-statements over the features — here, how
often each word appears in the description:

> **Decision Tree**
> - IF the word "TV" appears more than 3 times THEN
>   - IF the word "LED" appears more than 2 times THEN
>     - IF the word "HD" appears at least once THEN
>       - Price = \$500

Single trees overfit, so a Random Forest grows many of them — each on a random subset of the data
and features — and **averages** their predictions. We train 100 trees on a 15,000-row subset to
keep it quick (the full set is slow).

In [ ]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

In [ ]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [ ]:
evaluate(random_forest, test)

In [ ]:
# To persist a model (worth it if you train on the full dataset), use joblib:
# import joblib
# joblib.dump(rf_model, "random_forest.joblib")

## Traditional ML 4 — XGBoost

Like Random Forest, **XGBoost** is a tree ensemble — but instead of averaging independent trees,
it builds them **one after another**, with each new tree correcting the errors of the ones before
it (gradient boosting). It's fast and tends to generalize well, so we can comfortably train it on
the whole dataset.

> **If the import fails, skip this section — it's optional.** On a Mac you may first need
> `brew install libomp` in a terminal.

In [ ]:
import xgboost as xgb

In [ ]:
np.random.seed(42)

xgb_model = xgb.XGBRegressor(n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1)
xgb_model.fit(X, prices)

In [ ]:
def xg_boost(item):
    x = vectorizer.transform([item.summary])
    return max(0, xgb_model.predict(x)[0])

In [ ]:
evaluate(xg_boost, test)

### 💡 Business applications

Traditional ML isn't just history-class material — it's still heavily used in industry, especially
for tasks with clearly identifiable features. It's fast, cheap to run, and interpretable. It's
worth experimenting here: **see if you can beat these numbers with traditional ML.** For reference,
Ed ran the Random Forest over the full 800,000-row training set — it took ~15 hours and reached a
low error of about \$56.40. On Day 4 we'll see whether deep learning and LLMs can do better.